In [1]:
import os
import glob
import numpy as np
import pandas as pd




In [2]:
# BUGFIX: Make paths robust across Kaggle environments (sometimes /kaggle/input, sometimes ../input).
# This is I/O-only and score-neutral.
def _pick_first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


output_dir = "./"

test_dir = _pick_first_existing(
    [
        "/kaggle/input/plant-pathology-2021-fgvc8/test_images/",
        "../input/plant-pathology-2021-fgvc8/test_images/",
        "/kaggle/data/plant-pathology-2021-fgvc8/test_images/",
        "../data/plant-pathology-2021-fgvc8/test_images/",
    ]
)
if test_dir is None:
    raise FileNotFoundError(
        "Could not find test_images directory in expected locations."
    )

train_csv_path = _pick_first_existing(
    [
        "/kaggle/input/plant-pathology-2021-fgvc8/train.csv",
        "../input/plant-pathology-2021-fgvc8/train.csv",
        "/kaggle/data/plant-pathology-2021-fgvc8/train.csv",
        "../data/plant-pathology-2021-fgvcvc8/train.csv",
        "../data/plant-pathology-2021-fgvc8/train.csv",
    ]
)
if train_csv_path is None:
    raise FileNotFoundError("Could not find train.csv in expected locations.")

model_dir = _pick_first_existing(
    [
        "/kaggle/input/conve01/eff7-e9/epoch-9",
        "../input/conve01/eff7-e9/epoch-9",
    ]
)
# model_dir may be None; code below handles missing weights.

efficientB7 = _pick_first_existing(
    [
        "/kaggle/input/efficientb7/effb7",
        "../input/efficientb7/effb7",
    ]
)
# efficientB7 may be None; fallback uses keras.applications weights.

image_dims = (300, 300, 3)

data_set = pd.read_csv(train_csv_path)
df_labels = data_set["labels"].fillna("")
one_hot = df_labels.str.get_dummies(sep=" ")
dataset_labels = one_hot.columns.to_list()
NUM_CLASSES = len(dataset_labels)

print("NUM_CLASSES:", NUM_CLASSES)
print("Labels:", dataset_labels)




NUM_CLASSES: 6
Labels: ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab']


In [3]:
# NOTE: TensorFlow/Keras imports are intentionally moved here to avoid environment-level protobuf issues
# seen when importing TF at top-level in some Kaggle images (MessageFactory.GetPrototype crash).
def _find_saved_model_dir(path: str):
    """Return a directory containing a valid TF SavedModel (has saved_model.pb or saved_model.pbtxt)."""
    if not path or not os.path.exists(path):
        return None

    def is_savedmodel_dir(d):
        return os.path.isdir(d) and (
            os.path.exists(os.path.join(d, "saved_model.pb"))
            or os.path.exists(os.path.join(d, "saved_model.pbtxt"))
        )

    if is_savedmodel_dir(path):
        return path

    for cand in (
        os.path.join(path, "saved_model"),
        os.path.join(path, "export"),
        os.path.join(path, "1"),
    ):
        if is_savedmodel_dir(cand):
            return cand

    try:
        subdirs = [os.path.join(path, d) for d in os.listdir(path)]
    except Exception:
        subdirs = []
    for d in subdirs:
        if is_savedmodel_dir(d):
            return d

    for root, dirs, files in os.walk(path):
        if "saved_model.pb" in files or "saved_model.pbtxt" in files:
            return root
        rel_depth = os.path.relpath(root, path).count(os.sep)
        if rel_depth >= 1:
            dirs[:] = []
    return None




In [4]:
if __name__ == "__main__":
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import Sequential, Model
    from tensorflow.keras.layers import (
        Dense,
        BatchNormalization,
        Dropout,
        GlobalMaxPool2D,
        Conv2D,
        InputLayer,
    )

    class MultiLabel(Model):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)

            self.using_savedmodel = False
            savedmodel_dir = _find_saved_model_dir(efficientB7)

            self.model_backbone = None
            if savedmodel_dir is not None:
                try:
                    self.model_backbone = keras.layers.TFSMLayer(
                        savedmodel_dir, call_endpoint="serving_default"
                    )
                    self.using_savedmodel = True
                except Exception:
                    try:
                        self.model_backbone = keras.layers.TFSMLayer(
                            savedmodel_dir, call_endpoint="serve"
                        )
                        self.using_savedmodel = True
                    except Exception:
                        self.model_backbone = None
                        self.using_savedmodel = False

            if self.model_backbone is None:
                backbone = keras.applications.EfficientNetB7(
                    include_top=False,
                    weights="imagenet",
                    input_shape=image_dims,
                    pooling=None,
                )
                backbone.trainable = False
                self.model_backbone = backbone
                self.using_savedmodel = False

            self.model = Sequential()
            self.model.add(InputLayer(input_shape=image_dims))
            self.model.add(self.model_backbone)
            self.model.add(Conv2D(filters=1024, kernel_size=(1, 1), padding="same"))
            self.model.add(BatchNormalization(momentum=0.7))
            self.model.add(Dropout(0.2))
            self.model.add(Conv2D(filters=1024, kernel_size=(1, 1), padding="same"))
            self.model.add(
                Conv2D(filters=400 * NUM_CLASSES, kernel_size=(1, 1), padding="same")
            )

            self.pred_conv1 = Conv2D(filters=1024, kernel_size=(1, 1), padding="same")
            self.pred_bn = BatchNormalization()
            self.pred_conv2 = Conv2D(filters=512, kernel_size=(1, 1), padding="same")
            self.pred_pool = GlobalMaxPool2D()
            self.pred_dense = Dense(units=1, activation="sigmoid")

        def call(self, x, training=False, **kwargs):
            y = self.model(x, training=training)

            if isinstance(y, dict):
                if "outputs" in y:
                    y = y["outputs"]
                else:
                    y = next(iter(y.values()))

            shape = tf.shape(y)
            b, h, w = shape[0], shape[1], shape[2]

            y_per_class = tf.reshape(y, [b, h, w, NUM_CLASSES, 400])  # [B,H,W,C,400]
            y_per_class = tf.transpose(y_per_class, [0, 3, 1, 2, 4])  # [B,C,H,W,400]
            y_per_class = tf.reshape(
                y_per_class, [b * NUM_CLASSES, h, w, 400]
            )  # [B*C,H,W,400]

            # BUGFIX: pred_* layers must be built before accessing kernel/bias.
            # We ensure they are built once in the code below before first inference.
            k1, b1 = (self.pred_conv1.kernel, self.pred_conv1.bias)

            k1_g = tf.reshape(k1, [1, 1, 400, NUM_CLASSES, 1024])
            k1_g = tf.transpose(k1_g, [0, 1, 2, 4, 3])  # [1,1,400,1024,C]
            k1_g = tf.reshape(k1_g, [1, 1, 400, 1024 * NUM_CLASSES])

            z = tf.nn.conv2d(
                y_per_class,
                filters=k1_g,
                strides=[1, 1, 1, 1],
                padding="SAME",
            )  # [B*C,H,W,1024*C]
            z = tf.reshape(z, [b * NUM_CLASSES, h, w, 1024, NUM_CLASSES])
            z = tf.linalg.diag_part(z)  # [B*C,H,W,1024]
            z = tf.nn.bias_add(z, b1)
            z = self.pred_bn(z, training=training)
            z = self.pred_conv2(z)
            z = self.pred_pool(z)
            p = self.pred_dense(z)  # [B*C,1]

            p = tf.reshape(p, [b, NUM_CLASSES])  # [B,C]
            return p

        def create_model(self):
            return self.model

    tf.random.set_seed(42)
    np.random.seed(42)

    try:
        tf.config.experimental.enable_op_determinism(True)
    except Exception:
        pass

    try:
        tf.config.optimizer.set_jit(True)
    except Exception:
        pass

    try:
        tf.config.threading.set_intra_op_parallelism_threads(
            max(1, (os.cpu_count() or 4) // 2)
        )
        tf.config.threading.set_inter_op_parallelism_threads(2)
    except Exception:
        pass

    model = MultiLabel()
    model.build(input_shape=[None, image_dims[0], image_dims[1], image_dims[2]])

    # BUGFIX: Build pred_* head layers once with the correct shapes so .kernel/.bias exist.
    # This preserves architecture and is score-neutral (no computation changes other than making it runnable).
    dummy = tf.zeros([1, image_dims[0], image_dims[1], image_dims[2]], dtype=tf.float32)
    y0 = model.model(dummy, training=False)
    if isinstance(y0, dict):
        y0 = y0["outputs"] if "outputs" in y0 else next(iter(y0.values()))
    head_in_channels = int(y0.shape[-1]) if y0.shape[-1] is not None else 400
    model.pred_conv1.build([None, None, None, head_in_channels])
    model.pred_bn.build([None, None, None, 1024])
    model.pred_conv2.build([None, None, None, 1024])
    model.pred_dense.build([None, 512])

    # Now it is safe to run a full forward pass.
    _ = model(dummy, training=False)

    weights_path = model_dir
    if weights_path and os.path.isdir(weights_path):
        candidates = []
        candidates.extend(glob.glob(os.path.join(weights_path, "*.weights.h5")))
        candidates.extend(glob.glob(os.path.join(weights_path, "*.h5")))
        candidates.extend(glob.glob(os.path.join(weights_path, "ckpt*.index")))
        candidates.extend(glob.glob(os.path.join(weights_path, "ckpt*")))
        candidates = [c for c in candidates if os.path.isfile(c)]
        weights_path = max(candidates, key=os.path.getmtime) if candidates else None

    if weights_path is not None and os.path.exists(weights_path):
        try:
            model.load_weights(weights_path)
            print("Loaded weights:", weights_path)
        except Exception:
            ckpt_prefix = weights_path
            if ckpt_prefix.endswith(".index"):
                ckpt_prefix = ckpt_prefix[: -len(".index")]
            model.load_weights(ckpt_prefix)
            print("Loaded checkpoint weights:", ckpt_prefix)
    else:
        print("No external weights found; using backbone default weights (if any).")

    image_files = sorted(
        [
            os.path.join(test_dir, p)
            for p in os.listdir(test_dir)
            if p.lower().endswith(".jpg")
        ]
    )
    if len(image_files) == 0:
        raise RuntimeError(f"No .jpg files found in test_dir: {test_dir}")

    AUTOTUNE = tf.data.AUTOTUNE
    BATCH_SIZE = 64

    @tf.function(
        reduce_retracing=True,
        input_signature=[tf.TensorSpec(shape=(), dtype=tf.string)],
    )
    def _load_and_preprocess(path):
        input_img = tf.io.read_file(path)
        image = tf.io.decode_jpeg(input_img, channels=3, dct_method="INTEGER_FAST")
        image = tf.image.convert_image_dtype(image, dtype=tf.float32)  # [0,1]
        image = tf.image.resize(image, [image_dims[0], image_dims[1]])
        image = image * 255.0
        image.set_shape(image_dims)
        name = tf.strings.split(path, os.sep)[-1]
        return name, image

    options = tf.data.Options()
    options.experimental_deterministic = True
    try:
        options.experimental_optimization.apply_default_optimizations = True
        options.experimental_optimization.autotune_buffers = True
        options.experimental_optimization.autotune_cpu_budget = 0
        options.experimental_optimization.map_parallelization = True
        options.experimental_optimization.parallel_batch = True
    except Exception:
        pass

    files_tf = tf.constant(image_files)
    ds = tf.data.Dataset.from_tensor_slices(files_tf).with_options(options)
    ds = ds.map(_load_and_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.apply(tf.data.experimental.ignore_errors())
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)

    classes = np.asarray(dataset_labels, dtype=object)

    thr = 0.7

    @tf.function(
        reduce_retracing=True,
        input_signature=[
            tf.TensorSpec(
                shape=[None, image_dims[0], image_dims[1], image_dims[2]],
                dtype=tf.float32,
            )
        ],
        jit_compile=True,
    )
    def _predict(images):
        return model(images, training=False)

    C = NUM_CLASSES
    if C <= 0:
        raise ValueError("NUM_CLASSES computed as 0; check train.csv labels parsing.")
    if C > 15:
        # Safety: with this competition C is 12; keep as a guard for bitmask approach.
        raise ValueError(
            f"Unexpected NUM_CLASSES={C}; bitmask LUT may overflow uint16."
        )

    bit_weights = 1 << np.arange(C, dtype=np.uint16)  # [C]
    lut = np.empty(1 << C, dtype=object)
    for m in range(1 << C):
        if m == 0:
            lut[m] = "healthy"
        else:
            idx = np.flatnonzero((m & bit_weights) != 0)
            lut[m] = " ".join(classes[idx]) if len(idx) else "healthy"

    n = len(image_files)
    out_images = np.empty(n, dtype=object)
    out_labels = np.empty(n, dtype=object)
    pos = 0

    for names_b, images_b in ds:
        probs = _predict(images_b).numpy()  # [B, C] float
        mask = probs > thr  # [B, C] bool
        names_np = names_b.numpy().astype("U")  # bytes -> str

        codes = (mask.astype(np.uint16, copy=False) * bit_weights).sum(axis=1)  # [B]
        rows = lut[codes]  # ndarray object, [B]

        bsz = len(rows)
        out_images[pos : pos + bsz] = names_np
        out_labels[pos : pos + bsz] = rows
        pos += bsz

    if pos != n:
        out_images = out_images[:pos]
        out_labels = out_labels[:pos]

    csv_pd = pd.DataFrame({"image": out_images, "labels": out_labels})
    out_path = os.path.join(output_dir, "submission.csv")
    csv_pd.to_csv(out_path, index=False)
    print("Wrote submission.csv with shape:", csv_pd.shape, "to", out_path)
    print(csv_pd.head())

2026-05-12 23:22:54.556163: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 23:22:54.575111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778628174.597272      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778628174.601701      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 23:22:54.625563: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-05-12 23:23:09.766227: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'multi_label', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
2026-05-12 23:23:24.573780: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: INVALID_ARGUMENT: Must provide as many biases as the last dimension of the input tensor: [1024] vs. [6,10,10,6]


InvalidArgumentError: Exception encountered when calling MultiLabel.call().

[1m{{function_node __wrapped__BiasAdd_device_/job:localhost/replica:0/task:0/device:CPU:0}} Must provide as many biases as the last dimension of the input tensor: [1024] vs. [6,10,10,6] [Op:BiasAdd][0m

Arguments received by MultiLabel.call():
  • x=tf.Tensor(shape=(1, 300, 300, 3), dtype=float32)
  • training=False
  • kwargs=<class 'inspect._empty'>